# Load to dataframe from bronze

In [0]:
cards_df = spark.table("jrvs_databricks_fundamentals.bronze.cards_data")
transactions_df = spark.table("jrvs_databricks_fundamentals.bronze.transactions_data")
user_df = spark.table("jrvs_databricks_fundamentals.bronze.users_data")
mcc_df = spark.table("jrvs_databricks_fundamentals.bronze.mcc_codes")
fraud_df = spark.table("jrvs_databricks_fundamentals.bronze.fraud_labels")

## Transformation

In [ ]:
from pyspark.sql.functions import to_date, date_format, col,trim

silver_cards_df = (
    cards_df.dropDuplicates(["id"])
    .withColumnRenamed("id", "card_id")
    # keep MM/yyyy but validate it via date round-trip
    .withColumn("expires", date_format(to_date(col("expires"), "MM/yyyy"), "MM/yyyy"))
    .withColumn("acct_open_date", date_format(to_date(col("acct_open_date"), "MM/yyyy"), "MM/yyyy"))
    .withColumn("credit_limit", col("credit_limit").cast("decimal(19,2)"))
    # convert boolean (yes == true), "is" to column implies -> boolean type
    .withColumn("is_on_dark_web", col("card_on_dark_web") == "Yes")
    .drop("card_on_dark_web")
)



##Transaction Data

In [ ]:
from pyspark.sql.functions import col, lit, when, trim, lower, upper,to_timestamp, to_date, hour, date_format

silver_transactions_df = (
    transactions_df
    .dropDuplicates(["id"])
    .withColumn("amount", col("amount").cast("decimal(19,2)"))
    .withColumn("date", to_timestamp("date"))
    .withColumn(
        "use_chip",
        when(trim(col("use_chip")) == "Online Transaction", lit("online"))
        .when(trim(col("use_chip")) == "Chip Transaction", lit("chip"))
        .when(trim(col("use_chip")) == "Swipe Transaction", lit("swipe"))
        .otherwise(lower(trim(col("use_chip"))))
    )
    # online transactions: flag them, keep the nulls
    .withColumn(
        "is_online",
        (col("use_chip") == "online") | (upper(trim(col("merchant_city"))) == "ONLINE")
    )
    # non-online rows with missing geo -> map to 'unknown' instead of null
    .withColumn(
        "merchant_state",
        when(~col("is_online") & col("merchant_state").isNull(), lit("unknown"))
        .otherwise(col("merchant_state"))
    )
)

silver_transactions_df = silver_transactions_df

silver_transactions_df.printSchema()
display(silver_transactions_df.limit(10))

In [ ]:
from pyspark.sql.functions import col, when, lit

silver_user_df = (
    user_df
    .dropDuplicates(["id"])
    .withColumnRenamed("id", "user_id")
    # money: decimal(19,4) -> decimal(19,2)
    .withColumn("per_capita_income", col("per_capita_income").cast("decimal(19,2)"))
    .withColumn("yearly_income", col("yearly_income").cast("decimal(19,2)"))
    .withColumn("total_debt", col("total_debt").cast("decimal(19,2)"))


    # birth_month: valid 1-12, else null
    .withColumn(
        "birth_month",
        when((col("birth_month") < 1) | (col("birth_month") > 12), lit(None))
        .otherwise(col("birth_month"))
    )
    # birth_year: sane range, else null
    .withColumn(
        "birth_year",
        when((col("birth_year") < 1900), lit(None))
        .otherwise(col("birth_year"))
    )


)


silver_user_df.printSchema()
display(silver_user_df.limit(10))

In [ ]:
from pyspark.sql.functions import col

silver_mcc_df = (
    mcc_df
    .withColumn("mcc_code", col("mcc_code").cast("int"))
    .dropDuplicates(["mcc_code"])
)

silver_mcc_df = trim_strings(silver_mcc_df)
silver_mcc_df.printSchema()
display(silver_mcc_df.limit(10))

In [ ]:
from pyspark.sql.functions import col, trim, lower, when, lit

silver_fraud_df = (
    fraud_df
    # match transactions.id type for the join
    .withColumn("transaction_id", col("transaction_id").cast("int"))

    # Yes/No -> boolean is_fraud
    .withColumn(
        "is_fraud",
        when(lower(trim(col("label"))) == "yes", lit(True))
        .when(lower(trim(col("label"))) == "no", lit(False))
        .otherwise(lit(None).cast("boolean"))
    )
    .drop("label")
    .dropDuplicates(["transaction_id"])
)

silver_fraud_df.printSchema()
display(silver_fraud_df.limit(10))

In [ ]:
from pyspark.sql.functions import col, lit, when, trim, lower, upper, to_timestamp

silver_transactions_df = (
    transactions_df
    .withColumn("amount", col("amount").cast("decimal(19,2)"))
    .withColumn("date", to_timestamp("date"))
    .withColumn(
        "use_chip",
        when(trim(col("use_chip")) == "Online Transaction", lit("online"))
        .when(trim(col("use_chip")) == "Chip Transaction", lit("chip"))
        .when(trim(col("use_chip")) == "Swipe Transaction", lit("swipe"))
        .otherwise(lower(trim(col("use_chip"))))
    )
    # flag online transactions
    .withColumn(
        "is_online",
        (col("use_chip") == "online") | (upper(trim(col("merchant_city"))) == "ONLINE")
    )
    # non online rows with missing geo -> 'unknown' instead of null
    .withColumn(
        "merchant_state",
        when(~col("is_online") & col("merchant_state").isNull(), lit("unknown"))
        .otherwise(col("merchant_state"))
    )
    .withColumn(
        "zip",
        when(~col("is_online") & col("merchant_state").isNull(), lit("unknown"))
        .otherwise(col("zip"))
    )
    
)

silver_transactions_df.printSchema()
display(silver_transactions_df.limit(10))

## Save as Silver Tables

In [ ]:
CATALOG = "jrvs_databricks_fundamentals"

(silver_transactions_df.write.mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.silver.transactions_data"))

(silver_cards_df.write.mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.silver.cards_data"))

(silver_user_df.write.mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.silver.users_data"))

(silver_mcc_df.write.mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.silver.mcc_codes"))

(silver_fraud_df.write.mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.silver.fraud_labels"))